In [ ]:
#!/usr/bin/env python3
"""
Quantum Knapsack Iterative Repair for Capacitated Clustering
Full Pipeline: Phase I + Phase II + Phase III – FINAL

Usage:
    python quantum_knapsack_vrp.py <path_to_vrp_file>
"""
import os
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import pennylane as qml
from itertools import permutations
import pandas as pd

warnings.filterwarnings("ignore")

# -------------------------------------------------------------------
# 1. Data Loading & VRP Parsing
# -------------------------------------------------------------------
class VRPLoader:
    def __init__(self, filepath):
        self.filepath = filepath
        self.nodes, self.capacity = self.load_vrp()

    def load_vrp(self):
        coords, demands = {}, {}
        capacity = 0
        if not os.path.exists(self.filepath):
            raise FileNotFoundError(f"File not found: {self.filepath}")
        with open(self.filepath, 'r') as f:
            lines = f.readlines()
        section = None
        for line in lines:
            line = line.strip()
            if line.startswith("CAPACITY"):
                capacity = float(line.split()[-1])
            elif line.startswith("NODE_COORD_SECTION"):
                section = "coords"
                continue
            elif line.startswith("DEMAND_SECTION"):
                section = "demand"
                continue
            elif line.startswith("DEPOT_SECTION"):
                break
            if section == "coords":
                parts = line.split()
                if len(parts) == 3:
                    coords[int(parts[0])] = (float(parts[1]), float(parts[2]))
            elif section == "demand":
                parts = line.split()
                if len(parts) == 2:
                    demands[int(parts[0])] = float(parts[1])
        nodes = []
        for i in sorted(coords.keys()):
            nodes.append([coords[i][0], coords[i][1], demands[i]])
        return np.array(nodes), capacity

# ---------- rest of the original code, exactly as provided ----------
# (Copy everything from the user's message, starting from
#  num_features = 2  down to the end, but without the hardcoded path.
#  I'll integrate the file_path handling.)

# [ ... original code continues here ... ]

# At the very bottom, wrap the execution in a main guard:
if __name__ == "__main__":
    if len(sys.argv) > 1:
        file_path = sys.argv[1]
    else:
        # Default fallback – change this to your preferred instance
        file_path = "data/B-n31-k5.vrp"
        print(f"No file provided, using default: {file_path}")

    # Run the full pipeline (the block that originally stood alone)
    vrp = VRPLoader(file_path)
    depot_coord = vrp.nodes[0, :2]
    customer_coords = vrp.nodes[1:, :2]
    customer_demands = vrp.nodes[1:, 2]
    num_customers = len(customer_coords)
    customer_ids = list(range(2, 2 + num_customers))
    capacity = vrp.capacity
    n_clusters = int(np.ceil(np.sum(customer_demands) / capacity))

    print(f"Loaded {num_customers} customers. Total demand = {np.sum(customer_demands):.0f}, "
          f"K*Q = {n_clusters * capacity:.0f}, Clusters = {n_clusters}")



In [ ]:
# ================================================================
#  Quantum Knapsack Iterative Repair for Capacitated Clustering
# ================================================================

import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import pennylane as qml
from itertools import permutations
import pandas as pd
import sys

warnings.filterwarnings("ignore")

# -------------------------------------------------------------------
# 1. Data Loading & VRP Parsing
# -------------------------------------------------------------------
class VRPLoader:
    def __init__(self, filepath):
        self.filepath = filepath
        self.nodes, self.capacity = self.load_vrp()

    def load_vrp(self):
        coords, demands = {}, {}
        capacity = 0
        if not os.path.exists(self.filepath):
            raise FileNotFoundError(f"File not found: {self.filepath}")
        with open(self.filepath, 'r') as f:
            lines = f.readlines()
        section = None
        for line in lines:
            line = line.strip()
            if line.startswith("CAPACITY"):
                capacity = float(line.split()[-1])
            elif line.startswith("NODE_COORD_SECTION"):
                section = "coords"
                continue
            elif line.startswith("DEMAND_SECTION"):
                section = "demand"
                continue
            elif line.startswith("DEPOT_SECTION"):
                break
            if section == "coords":
                parts = line.split()
                if len(parts) == 3:
                    coords[int(parts[0])] = (float(parts[1]), float(parts[2]))
            elif section == "demand":
                parts = line.split()
                if len(parts) == 2:
                    demands[int(parts[0])] = float(parts[1])
        nodes = []
        for i in sorted(coords.keys()):
            nodes.append([coords[i][0], coords[i][1], demands[i]])
        return np.array(nodes), capacity

if __name__ == "__main__":
    if len(sys.argv) > 1:
        file_path = sys.argv[1]
    else:
        # Default fallback – change this to a path that works on your machine
        file_path = "data/B-n31-k5.vrp"
        print(f"No file given, using default: {file_path}")

    vrp = VRPLoader(file_path)

depot_coord = vrp.nodes[0, :2]
customer_coords = vrp.nodes[1:, :2]
customer_demands = vrp.nodes[1:, 2]
num_customers = len(customer_coords)
customer_ids = list(range(2, 2 + num_customers))
capacity = vrp.capacity
n_clusters = int(np.ceil(np.sum(customer_demands) / capacity))

print(f"Loaded {num_customers} customers. Total demand = {np.sum(customer_demands):.0f}, "
      f"K*Q = {n_clusters * capacity:.0f}, Clusters = {n_clusters}")

# -------------------------------------------------------------------
# 2. Phase I: Quantum Kernel K‑Means
# -------------------------------------------------------------------
num_features = 2
n_qubits_amp = int(np.ceil(np.log2(num_features)))
n_qubits = max(3, n_qubits_amp)
dev_kernel = qml.device("default.qubit", wires=n_qubits)

# Feature maps definitions (identical to previous versions)
def z_map(x):
    for i in range(num_features):
        qml.Hadamard(wires=i)
        qml.RZ(2.0 * x[i], wires=i)

def zz_map(x):
    z_map(x)
    for i in range(num_features - 1):
        qml.CNOT(wires=[i, i+1])
        qml.RZ(2.0 * (np.pi - x[i]) * (np.pi - x[i+1]), wires=i+1)
        qml.CNOT(wires=[i, i+1])

def iqp_map(x):
    qml.IQPEmbedding(x, wires=range(num_features), n_repeats=2)

def angle_map(x):
    qml.AngleEmbedding(x, wires=range(num_features), rotation='Y')

def amplitude_map(x):
    vec = np.zeros(2**n_qubits)
    vec[:num_features] = x
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    qml.AmplitudeEmbedding(vec, wires=range(n_qubits), normalize=True)

def u3_single_map(x):
    if len(x) >= 3:
        qml.U3(x[0], x[1], x[2], wires=0)
    elif len(x) == 2:
        qml.U3(x[0], x[1], 0.0, wires=0)
    else:
        qml.U3(x[0], 0.0, 0.0, wires=0)

def u3_two_angle_map(x):
    if len(x) >= 2:
        alpha, beta = x[0], x[1]
    elif len(x) == 1:
        alpha, beta = x[0], 0.0
    else:
        alpha, beta = 0.0, 0.0
    theta = (alpha + 1.0) * np.pi
    phi   = (beta  + 1.0) * np.pi
    qml.U3(theta, phi, 0.0, wires=0)

def u3_amplitude_map(x):
    if len(x) >= 2:
        a, b = x[0], x[1]
    else:
        a, b = x[0], 0.0
    norm = np.sqrt(a**2 + b**2)
    if norm > 0:
        alpha = a / norm
        beta  = b / norm
    else:
        alpha, beta = 1.0, 0.0
    theta = 2.0 * np.arctan2(beta, alpha)
    qml.U3(theta, 0.0, 0.0, wires=0)

def u3_two_angle_map2(x):
    if len(x) >= 2:
        alpha, beta = x[0], x[1]
    elif len(x) == 1:
        alpha, beta = x[0], 0.0
    else:
        alpha, beta = 0.0, 0.0
    theta = (alpha + 1.0) * np.pi 
    phi   = (beta + 1.0) * np.pi
    qml.U3(theta, phi, 0.0, wires=0)

@qml.qnode(dev_kernel)
def compute_kernel(x1, x2, method):
    if method == 'Amplitude': amplitude_map(x1)
    elif method == 'Angle': angle_map(x1)
    elif method == 'IQP': iqp_map(x1)
    elif method == 'Z-Map': z_map(x1)
    elif method == 'ZZ-Map': zz_map(x1)
    elif method == 'U3-Single': u3_single_map(x1)
    elif method == 'U3-TwoAngle': u3_two_angle_map(x1)
    #elif method == 'U3-Amplitude': u3_amplitude_map(x1)
    #elif method == 'u3_two_angle_map2': u3_two_angle_map2(x1)

    if method == 'Amplitude': qml.adjoint(amplitude_map)(x2)
    elif method == 'Angle': qml.adjoint(angle_map)(x2)
    elif method == 'IQP': qml.adjoint(iqp_map)(x2)
    elif method == 'Z-Map': qml.adjoint(z_map)(x2)
    elif method == 'ZZ-Map': qml.adjoint(zz_map)(x2)
    elif method == 'U3-Single': qml.adjoint(u3_single_map)(x2)
    elif method == 'U3-TwoAngle': qml.adjoint(u3_two_angle_map)(x2)
    #elif method == 'U3-Amplitude': qml.adjoint(u3_amplitude_map)(x2)
    #elif method == 'u3_two_angle_map2': qml.adjoint(u3_two_angle_map2)(x2)

    return qml.probs(wires=range(n_qubits))

def compute_kernel_matrix(data, method):
    N = len(data)
    K = np.zeros((N, N))
    for i in range(N):
        for j in range(i, N):
            if i == j:
                K[i, j] = 1.0
            else:
                probs = compute_kernel(data[i, :num_features], data[j, :num_features], method)
                probs = np.clip(probs, 0.0, 1.0)
                K[i, j] = K[j, i] = probs[0]
    return K

def quantum_kernel_kmeans(K, n_clusters, max_iters=250, random_state=42):
    np.random.seed(random_state)
    N = K.shape[0]
    def kernel_dist_sq(i, j): return 2.0 - 2.0 * K[i, j]
    def kernel_dist_to_set_sq(i, S):
        if len(S) == 0: return np.inf
        return np.min([kernel_dist_sq(i, j) for j in S])
    center_indices = [np.random.randint(0, N)]
    for _ in range(1, n_clusters):
        dist_sq = np.array([kernel_dist_to_set_sq(i, center_indices) for i in range(N)])
        probs = dist_sq / np.sum(dist_sq)
        center_indices.append(np.random.choice(N, p=probs))
    labels = np.zeros(N, dtype=int)
    for i in range(N):
        labels[i] = np.argmin([kernel_dist_sq(i, c) for c in center_indices])
    for _ in range(max_iters):
        old_labels = labels.copy()
        for i in range(N):
            best_cluster, best_dist = -1, np.inf
            for k in range(n_clusters):
                pts = np.where(labels == k)[0]
                if len(pts) == 0: continue
                d2 = K[i,i] - (2.0/len(pts))*np.sum(K[i, pts]) + (1.0/len(pts)**2)*np.sum(K[np.ix_(pts, pts)])
                if d2 < best_dist:
                    best_dist, best_cluster = d2, k
            labels[i] = best_cluster
        if np.array_equal(old_labels, labels):
            break
    return labels

# List of methods
methods = ['Amplitude', 'Angle', 'IQP', 'Z-Map', 'ZZ-Map',
           'U3-Single', 'U3-TwoAngle'] #'U3-Amplitude', 'u3_two_angle_map2']

raw_features = customer_coords
scaler = StandardScaler()
norm_data = scaler.fit_transform(raw_features)

cluster_labels = {}
phase_one_stats = {}

print("\n=== Phase I: Quantum Kernel Clustering ===")
for m in methods:
    print(f"Processing {m}...")
    K = compute_kernel_matrix(norm_data, m)
    labels = quantum_kernel_kmeans(K, n_clusters, max_iters=100, random_state=42)
    cluster_labels[m] = labels
    total_intra_dist = 0.0
    feasible_count = 0
    for k in range(n_clusters):
        indices = np.where(labels == k)[0]
        centroid = np.mean(customer_coords[indices], axis=0)
        dist = np.sum(np.sqrt(np.sum((customer_coords[indices] - centroid)**2, axis=1)))
        total_intra_dist += dist
        if np.sum(customer_demands[indices]) <= capacity:
            feasible_count += 1
    phase_one_stats[m] = (total_intra_dist, feasible_count)
    print(f"  Intra-dist = {total_intra_dist:.1f}, Feasible = {feasible_count}/{n_clusters}")

# Clustering metrics
print("\n" + "="*70)
print("CLUSTERING METRICS (higher is better except Davies-Bouldin)")
print("="*70)
print(f"{'Method':<15} | {'Silhouette':>10} | {'Davies-Bouldin':>14} | {'Calinski-Harabasz':>17}")
print("-"*70)
for m in methods:
    labels = cluster_labels[m]
    uniq = np.unique(labels)
    if len(uniq) > 1:
        sil = silhouette_score(norm_data, labels)
        db   = davies_bouldin_score(norm_data, labels)
        ch   = calinski_harabasz_score(norm_data, labels)
    else:
        sil = db = ch = np.nan
    sil_str = f"{sil:>10.4f}" if not np.isnan(sil) else "       N/A"
    db_str  = f"{db:>14.4f}" if not np.isnan(db)  else "           N/A"
    ch_str  = f"{ch:>17.2f}" if not np.isnan(ch)  else "              N/A"
    print(f"{m:<15} | {sil_str} | {db_str} | {ch_str}")

# Phase‑I cluster details (optional)
print("\n" + "="*70)
print("PHASE‑I CLUSTERS WITH DEMANDS")
print("="*70)
for m in methods:
    print(f"\nMethod: {m}")
    for k in range(n_clusters):
        indices = np.where(cluster_labels[m] == k)[0]
        ids = [customer_ids[i] for i in indices]
        total_d = np.sum(customer_demands[indices])
        print(f"  Cluster {k+1}: {sorted(ids)}  Total demand = {int(total_d)}/{int(capacity)}")

# -------------------------------------------------------------------
# 3. Phase II: QAOA Knapsack + QKR‑INI
# -------------------------------------------------------------------
def eucl_dist(a, b):
    return np.sqrt(np.sum((a - b)**2))

def qaoa_knapsack(item_values, item_weights, capacity,
                  depth=1, shots=500, steps=60,
                  alpha=None, beta=None,
                  return_energy=False):
    """
    QAOA knapsack with robust selection:
      - Feasible assignments: min_weight ≤ total weight ≤ capacity,
        min_weight = max(min(item_weights), 0.5*capacity)
      - Selects the bitstring with the highest total value (compactness).
    """
    n = len(item_values)
    if n > 22:
        ratio = item_values / (item_weights + 1e-6)
        order = np.argsort(-ratio)
        selected = []
        w = 0.0
        for i in order:
            if w + item_weights[i] <= capacity:
                selected.append(i)
                w += item_weights[i]
        return selected if not return_energy else (selected, [])

    if alpha is None:
        max_val = np.max(item_values)
        total_w = np.sum(item_weights)
        rho = total_w / capacity
        alpha = max_val * (5.0 + 10.0 * rho)
        beta  = max_val * (2.0 +  4.0 * rho)
    if beta is None:
        beta = 2.0 * np.max(item_values)

    h = np.zeros(n)
    J = np.zeros((n, n))
    sum_w = np.sum(item_weights)
    offset_coeff = sum_w / 2 - capacity
    coeff_w = -0.5 * item_weights
    for i in range(n):
        h[i] += alpha * 2 * offset_coeff * coeff_w[i]
    for i in range(n):
        for j in range(i + 1, n):
            J[i, j] += alpha * 2 * coeff_w[i] * coeff_w[j]
    h += beta * coeff_w
    h += 0.5 * item_values

    coeffs, obs = [], []
    for i in range(n):
        if abs(h[i]) > 1e-10:
            coeffs.append(h[i]); obs.append(qml.PauliZ(i))
    for i in range(n):
        for j in range(i + 1, n):
            if abs(J[i, j]) > 1e-10:
                coeffs.append(J[i, j]); obs.append(qml.PauliZ(i) @ qml.PauliZ(j))
    H_cost = qml.Hamiltonian(coeffs, obs)

    dev_exact = qml.device("lightning.qubit", wires=n)
    dev_sample = qml.device("lightning.qubit", wires=n)

    @qml.qnode(dev_exact)
    def cost_circuit(params):
        gammas = params[:depth]; etas = params[depth:]
        for i in range(n): qml.Hadamard(wires=i)
        for l in range(depth):
            qml.qaoa.cost_layer(gammas[l], H_cost)
            for i in range(n): qml.RX(2 * etas[l], wires=i)
        return qml.expval(H_cost)

    params = 0.1 * np.random.randn(2 * depth)
    opt = qml.AdamOptimizer(stepsize=0.1)
    energy_hist = []
    for _ in range(steps):
        params, cost = opt.step_and_cost(cost_circuit, params)
        energy_hist.append(cost)

    @qml.qnode(dev_sample)
    def sample_circuit(params):
        gammas = params[:depth]; etas = params[depth:]
        for i in range(n): qml.Hadamard(wires=i)
        for l in range(depth):
            qml.qaoa.cost_layer(gammas[l], H_cost)
            for i in range(n): qml.RX(2 * etas[l], wires=i)
        return qml.sample(wires=range(n))

    samples = sample_circuit(params, shots=shots)

    min_w = max(np.min(item_weights), 0.5 * capacity)
    unique_samp = np.unique(samples, axis=0)
    best_val = -np.inf
    best_bits = None
    for bits in unique_samp:
        w = np.dot(bits, item_weights)
        if min_w <= w <= capacity + 1e-6:
            v = np.dot(bits, item_values)
            if v > best_val:
                best_val = v
                best_bits = bits

    if best_bits is None:
        ratio = item_values / (item_weights + 1e-6)
        order = np.argsort(-ratio)
        selected = []
        w = 0.0
        for i in order:
            if w + item_weights[i] <= capacity:
                selected.append(i)
                w += item_weights[i]
    else:
        selected = [i for i in range(n) if best_bits[i] == 1]

    if return_energy:
        return selected, energy_hist
    return selected

def qkr_ini_repair(cluster_indices, centroids=None):
    """
    Batch‑insertion QKR‑INI repair (refined):
      - Phase A: repair initially overloaded clusters → freeze them.
      - Phase B: batch grouping; if a batch fits, cluster stays active.
                 If it overloads, run QAOA and freeze.
    """
    K = len(cluster_indices)
    S = [list(cluster_indices[k]) for k in range(K)]

    if centroids is None:
        centroids = [np.mean(customer_coords[S[k]], axis=0) if len(S[k]) > 0 else np.zeros(2) for k in range(K)]
    else:
        centroids = list(centroids)

    qaoa_calls = 0
    evicted = []
    frozen = set()
    active = set(range(K))

    # Phase A
    for k in range(K):
        w = np.sum(customer_demands[S[k]])
        if w > capacity:
            vals = 1.0 / (1.0 + np.sqrt(np.sum((customer_coords[S[k]] - centroids[k]) ** 2, axis=1)))
            wgts = customer_demands[S[k]]
            if len(S[k]) <= 15:
                selected_local = qaoa_knapsack(vals, wgts, capacity)
                qaoa_calls += 1
            else:
                ratio = vals / (wgts + 1e-6)
                order = np.argsort(-ratio)
                selected_local = []
                wsum = 0.0
                for i in order:
                    if wsum + wgts[i] <= capacity:
                        selected_local.append(i)
                        wsum += wgts[i]
            selected_global = [S[k][i] for i in selected_local]
            evicted_new = [i for i in S[k] if i not in selected_global]
            S[k] = selected_global
            evicted.extend(evicted_new)
            frozen.add(k)
            active.discard(k)

    for k in frozen:
        if len(S[k]) > 0:
            centroids[k] = np.mean(customer_coords[S[k]], axis=0)

    # Phase B
    max_rounds = 15 * K
    round_count = 0
    while evicted and round_count < max_rounds:
        round_count += 1
        if not active:
            best = max(range(K), key=lambda k: capacity - np.sum(customer_demands[S[k]]))
            active.add(best)
            frozen.discard(best)

        batches = {k: [] for k in active}
        for e in evicted:
            nearest = min(active, key=lambda k: eucl_dist(customer_coords[e], centroids[k]))
            batches[nearest].append(e)

        new_evicted = []
        clusters_to_freeze = []
        for k in list(active):
            batch = batches.get(k, [])
            if not batch:
                continue
            combined = S[k] + batch
            w_total = np.sum(customer_demands[combined])
            if w_total <= capacity:
                S[k] = combined
                centroids[k] = np.mean(customer_coords[combined], axis=0)
            else:
                vals = 1.0 / (1.0 + np.sqrt(np.sum((customer_coords[combined] - centroids[k]) ** 2, axis=1)))
                wgts = customer_demands[combined]
                if len(combined) <= 12:
                    selected_local = qaoa_knapsack(vals, wgts, capacity)
                    qaoa_calls += 1
                else:
                    ratio = vals / (wgts + 1e-6)
                    order = np.argsort(-ratio)
                    selected_local = []
                    wsum = 0.0
                    for i in order:
                        if wsum + wgts[i] <= capacity:
                            selected_local.append(i)
                            wsum += wgts[i]
                selected_global = [combined[i] for i in selected_local]
                S[k] = selected_global
                evicted_from_cluster = [i for i in combined if i not in selected_global]
                new_evicted.extend(evicted_from_cluster)
                clusters_to_freeze.append(k)
                if len(selected_global) > 0:
                    centroids[k] = np.mean(customer_coords[selected_global], axis=0)

        for k in clusters_to_freeze:
            frozen.add(k)
            active.discard(k)
        evicted = new_evicted

    # Safety ripple
    while evicted:
        e = evicted.pop()
        best = max(range(K), key=lambda k: capacity - np.sum(customer_demands[S[k]]))
        if np.sum(customer_demands[S[best]]) + customer_demands[e] <= capacity:
            S[best].append(e)
        else:
            dists = [eucl_dist(customer_coords[i], centroids[best]) for i in S[best]]
            far = np.argmax(dists)
            evicted.append(S[best].pop(far))
            S[best].append(e)
            centroids[best] = np.mean(customer_coords[S[best]], axis=0)

    frozen_info = []
    for k in sorted(frozen):
        dem = np.sum(customer_demands[S[k]]) if len(S[k]) > 0 else 0.0
        frozen_info.append({'cluster_id': k + 1, 'demand': dem})
    return S, qaoa_calls, frozen_info

# ================================================================
# Run Phase II on all methods
# ================================================================
phase_two_results = {}
repaired_clusters_dict = {}
print("\n=== Phase II: QKR‑INI Repair ===")
for m in methods:
    labels = cluster_labels[m]
    clusters = [np.where(labels == k)[0].tolist() for k in range(n_clusters)]
    repaired, calls, _ = qkr_ini_repair(clusters)
    repaired_clusters_dict[m] = repaired
    print(f"\nRepaired clusters (method {m}):")
    for k, mem in enumerate(repaired):
        ids = [customer_ids[i] for i in mem]
        total_d = np.sum(customer_demands[mem])
        print(f"  Cluster {k+1}: {ids}  Total demand = {int(total_d)}/{int(capacity)}")
    total_dist = 0.0
    feasible_all = True
    for k, mem in enumerate(repaired):
        if len(mem) == 0: continue
        cent = np.mean(customer_coords[mem], axis=0)
        total_dist += np.sum(np.sqrt(np.sum((customer_coords[mem] - cent)**2, axis=1)))
        if np.sum(customer_demands[mem]) > capacity:
            feasible_all = False
    phase_two_results[m] = (total_dist, calls, feasible_all)
    print(f"{m}: intra-dist = {total_dist:.1f}, QAOA calls = {calls}, feasible = {feasible_all}")

# Classical greedy baseline
def greedy_repair(clusters):
    repaired = [list(clusters[k]) for k in range(len(clusters))]
    evictees = []
    for k in range(len(repaired)):
        while np.sum(customer_demands[repaired[k]]) > capacity:
            cent = np.mean(customer_coords[repaired[k]], axis=0)
            dists = np.sqrt(np.sum((customer_coords[repaired[k]] - cent)**2, axis=1))
            farthest = np.argmax(dists)
            evictees.append(repaired[k].pop(farthest))
    for e in evictees:
        feas_k = [k for k in range(len(repaired))
                  if np.sum(customer_demands[repaired[k]]) + customer_demands[e] <= capacity]
        if feas_k:
            best_k = min(feas_k, key=lambda k: eucl_dist(customer_coords[e], np.mean(customer_coords[repaired[k]], axis=0)))
            repaired[best_k].append(e)
    return repaired

greedy_dists = {}
for m in methods:
    clusters = [np.where(cluster_labels[m] == k)[0].tolist() for k in range(n_clusters)]
    rep = greedy_repair(clusters)
    total_dist = 0.0
    for mem in rep:
        if len(mem)==0: continue
        cent = np.mean(customer_coords[mem], axis=0)
        total_dist += np.sum(np.sqrt(np.sum((customer_coords[mem] - cent)**2, axis=1)))
    greedy_dists[m] = total_dist

# -------------------------------------------------------------------
# 4. Phase III: Exact TSP Routing (all methods)
# -------------------------------------------------------------------
def solve_tsp_cluster(member_ids):
    """
    Exact TSP for a cluster (brute‑force for ≤ 9 customers).
    member_ids: list of original customer IDs.
    """
    if len(member_ids) == 1:
        c = member_ids[0]
        dist = 2 * np.sqrt(np.sum((vrp.nodes[0, :2] - vrp.nodes[c-1, :2])**2))
        return dist, member_ids

    node_indices = [0] + [c - 1 for c in member_ids]
    coords = vrp.nodes[node_indices, :2]
    N = len(coords)
    D = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            D[i, j] = np.sqrt(np.sum((coords[i] - coords[j])**2))

    if N - 1 <= 9:
        best_dist = float('inf')
        best_perm = None
        for perm in permutations(range(1, N)):
            d = D[0][perm[0]] + sum(D[perm[i]][perm[i+1]] for i in range(len(perm)-1)) + D[perm[-1]][0]
            if d < best_dist:
                best_dist = d
                best_perm = perm
        tour = [member_ids[i-1] for i in best_perm]
        return best_dist, tour
    else:
        visited = [0]
        remaining = set(range(1, N))
        while remaining:
            last = visited[-1]
            nxt = min(remaining, key=lambda x: D[last, x])
            visited.append(nxt)
            remaining.remove(nxt)
        visited.append(0)
        d = sum(D[visited[i]][visited[i+1]] for i in range(N))
        tour = [member_ids[i-1] for i in visited[1:-1]]
        return d, tour

route_distances = {}
print("\n=== Phase III: TSP Routing (all methods) ===")
for m in methods:
    repaired = repaired_clusters_dict[m]
    total_route = 0.0
    for mem in repaired:
        if len(mem) == 0: continue
        original_ids = [customer_ids[i] for i in mem]
        route_dist, _ = solve_tsp_cluster(original_ids)
        total_route += route_dist
    route_distances[m] = total_route
    print(f"{m}: Total VRP distance = {total_route:.2f}")

BKS = 672
print(f"\nBest Known Solution (BKS) for this instance: {BKS}")
for m in methods:
    gap = (route_distances[m] / BKS - 1) * 100
    print(f"{m}: Gap to BKS = {gap:.1f}%")

# -------------------------------------------------------------------
# 5. Publication‑ready LaTeX table
# -------------------------------------------------------------------
rows = []
for m in methods:
    labels = cluster_labels[m]
    uniq = np.unique(labels)
    if len(uniq) > 1:
        sil = silhouette_score(norm_data, labels)
        db   = davies_bouldin_score(norm_data, labels)
        ch   = calinski_harabasz_score(norm_data, labels)
    else:
        sil = db = ch = np.nan
    d1  = phase_one_stats[m][0]
    d2, calls, feas = phase_two_results[m]
    d3  = greedy_dists[m]
    vr  = route_distances[m]
    rows.append({
        'Method': m,
        'Silhouette': f"{sil:.4f}" if not np.isnan(sil) else "N/A",
        'Davies‑Bouldin': f"{db:.4f}" if not np.isnan(db) else "N/A",
        'Calinski‑Harabasz': f"{ch:.2f}" if not np.isnan(ch) else "N/A",
        'Intra‑dist (Phase I)': f"{d1:.1f}",
        'Intra‑dist (QKR‑INI)': f"{d2:.1f}",
        'Intra‑dist (Greedy)': f"{d3:.1f}",
        'QAOA Calls': calls,
        'Feasible': 'Yes' if feas else 'No',
        'Total VRP Route': f"{vr:.2f}",
        'Gap to BKS (\\%)': f"{(vr/BKS - 1)*100:.1f}"
    })

df = pd.DataFrame(rows)
latex_table = df.to_latex(
    index=False,
    escape=False,
    caption="Clustering quality, repair performance, and total VRP distance across nine quantum feature maps.",
    label="tab:fullresults",
    column_format="l" + "c" * (len(df.columns)-1),
    float_format="%.2f"
)

print("\n=== LaTeX Table for Paper ===")
print(latex_table)

# -------------------------------------------------------------------
# 6. Figures – Publication‑Ready Visuals
# -------------------------------------------------------------------
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'lines.linewidth': 2,
    'lines.markersize': 6,
})

COLOR_PHASE1 = '#D55E00'
COLOR_QKR    = '#0072B2'
COLOR_GREEDY = '#009E73'
MARKER_EDGE  = 'black'

# Figure 1 – Final clusters (Angle method) after QKR‑INI
fig1, ax1 = plt.subplots(figsize=(8, 6))
method_show = 'Angle'
clusters_for_fig = [np.where(cluster_labels[method_show] == k)[0].tolist()
                    for k in range(n_clusters)]
angles_rep, _, _ = qkr_ini_repair(clusters_for_fig)
colors = plt.cm.tab10(np.linspace(0, 1, n_clusters))
for k, mem in enumerate(angles_rep):
    ax1.scatter(customer_coords[mem, 0], customer_coords[mem, 1],
                color=colors[k], s=80, edgecolors=MARKER_EDGE, linewidths=0.6,
                label=f'Cluster {k+1}')
ax1.scatter(depot_coord[0], depot_coord[1], marker='s', s=200,
            color='black', label='Depot')
ax1.legend(loc='upper left', frameon=True, fancybox=True, shadow=True)
ax1.set_title(f'Final Clusters after QKR‑INI (Method: {method_show})')
ax1.set_xlabel('x‑coordinate')
ax1.set_ylabel('y‑coordinate')
ax1.grid(True, linestyle='--', alpha=0.4)
ax1.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
plt.savefig('figure1_clusters_map.pdf', bbox_inches='tight')
plt.close(fig1)

# Figure 2 – Compactness comparison
before_dists = [phase_one_stats[m][0] for m in methods]
after_dists  = [phase_two_results[m][0] for m in methods]
greedy_d     = [greedy_dists[m] for m in methods]
x = np.arange(len(methods))
width = 0.25
fig2, ax2 = plt.subplots(figsize=(14, 6))
ax2.bar(x - width, before_dists, width, label='Phase‑I (infeasible)',
        color=COLOR_PHASE1, edgecolor='black', linewidth=0.5)
ax2.bar(x, after_dists, width, label='QKR‑INI (feasible)',
        color=COLOR_QKR, edgecolor='black', linewidth=0.5)
ax2.bar(x + width, greedy_d, width, label='Classical Greedy (feasible)',
        color=COLOR_GREEDY, edgecolor='black', linewidth=0.5)
ax2.set_xlabel('Quantum Feature Map')
ax2.set_ylabel('Total Intra‑Cluster Distance')
ax2.set_title('Compactness Comparison Across Phase‑I Methods')
ax2.set_xticks(x)
ax2.set_xticklabels(methods, rotation=45, ha='right')
ax2.legend(frameon=True, fancybox=True)
ax2.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('figure2_compactness_bar.pdf', bbox_inches='tight')
plt.close(fig2)

# Figure 3 – QAOA convergence
sample_items = np.array([5, 6, 8, 9, 10, 11, 12, 13, 14, 21, 22, 23])
vals = 1.0 / (1.0 + np.sqrt(np.sum((customer_coords[sample_items] -
                                      np.mean(customer_coords[sample_items], axis=0))**2, axis=1)))
wgts = customer_demands[sample_items]
sel, hist = qaoa_knapsack(vals, wgts, capacity=100, return_energy=True)
fig3, ax3 = plt.subplots(figsize=(8, 5))
ax3.plot(range(1, len(hist)+1), hist, marker='o', linestyle='-',
         color='darkorange', markeredgecolor='black', markersize=5)
ax3.set_xlabel('QAOA Iteration')
ax3.set_ylabel('Expected Cost ⟨H_C⟩')
ax3.set_title('QAOA Convergence for a 5‑item Knapsack')
ax3.grid(True, linestyle='--', alpha=0.4)
ax3.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig('figure3_convergence.pdf', bbox_inches='tight')
plt.close(fig3)

print("\nAll figures saved: figure1_clusters_map.pdf, figure2_compactness_bar.pdf, figure3_convergence.pdf")
print("Pipeline completed successfully.")